# Fine-tune EFM for cell-type annotation

This workflow fine-tunes EFM and a linear classifier on a preprocessed, labelled AnnData file. SFM stays frozen and supplies the causal gene order. The input label column is split deterministically into 70% train and 30% held-out test cells.

In [ ]:
from pathlib import Path

from sccafm import cell_type_annotation

repo_root = Path.cwd()
if not (repo_root / 'configs').exists():
    repo_root = repo_root.parent

input_h5ad = Path('/path/to/preprocessed_labelled_cells.h5ad')
label_key = 'cell_type'
output_dir = repo_root / 'results' / 'efm_cell_type_annotation'
config_path = repo_root / 'configs' / 'efm_cell_type_annotation.yaml'
model_source = repo_root / 'assets'


The AnnData must already be preprocessed, have finite numeric `X`, contain only model-vocabulary genes, and have a non-null categorical label for every cell. Each label must occur in both the deterministic train and test partitions.

In [ ]:
run = cell_type_annotation.prepare(
    input_h5ad=input_h5ad,
    model_source=model_source,
    config_path=config_path,
    label_key=label_key,
)

print('Classes:', run.class_names)
print('Train cells:', len(run.train_indices))
print('Test cells:', len(run.test_indices))


In [ ]:
result = cell_type_annotation.fit(run, output_dir=output_dir)
print(result.to_summary())


In [ ]:
predictions = cell_type_annotation.predict(run)
predictions.head()


The output directory contains copied SFM/tokenizer assets, the fine-tuned EFM and classifier weights, the label map, split manifest, resumable train state, `metrics.json`, and `test_predictions.csv`.